# Training an ADMET model end-to-end with the OpenADMET stack

<img src="./img/the_caco2.png" width="600">

*Dr Maria Castellanos*

## OMSF 2026 Demo

Hi all, and welcome to the 2026 OpenADMET Demo! This notebook walks through our full stack end-to-end using **LogD** as a worked example:

1. **Train** a CheMeleon-based GNN on curated ChEMBL data using **Anvil**, our reproducible model-training harness
2. **Run inference** on a real program test set with `openadmet predict`
3. **Interact naturally** with the entire workflow using our **Claude AI skill**
4. **Get predictions instantly** via our HuggingFace **Prediction Portal** — no code needed
5. **Deploy internally** inside your organisation using Docker

This is adapted from the tutorials at our [demos repo](https://demos.openadmet.org). Check it out for deeper dives into fine-tuning, applicability domain, and more.

## Why LogD?

**LogD** is the logarithm of the distribution coefficient — it measures how a compound partitions between octanol and water at physiological pH (typically 7.4). Unlike the simpler LogP (which ignores ionisation), LogD accounts for the ionisation state of the molecule, making it the more relevant descriptor for biological systems.

LogD sits at the centre of the **lipophilicity** story in drug discovery:

* **Solubility**: high LogD → low aqueous solubility → poor oral absorption
* **Membrane permeability**: LogD is a strong predictor of passive diffusion across lipid bilayers
* **Protein binding**: lipophilic compounds bind non-specifically to plasma proteins (albumin, alpha-1 acid glycoprotein), reducing free drug concentration
* **Metabolic liability**: highly lipophilic compounds are preferential CYP450 substrates, shortening half-life
* **Central nervous system penetration**: blood-brain barrier crossing is strongly correlated with LogD

The "Lipophilicity Efficiency" (LipE / LLE = pIC50 − LogD) has become a widely used compass in lead optimisation — maximising potency while keeping lipophilicity low is one of the clearest routes to reducing ADMET attrition.

LogD is also considered one of the more **learnable** ADMET endpoints. Because lipophilicity arises largely from additive fragment contributions (alkyl chains, aromatic rings, polar groups), a good molecular representation can capture much of the variance from structure alone. This makes it a perfect first endpoint to explore with a GNN.

<img src="img/expansion.png" width="600">

We will use the **ExpansionRx dataset** as our benchmark — the largest public ADMET dataset released from a real drug discovery program (Expansion Therapeutics). It covers more than 7,500 compounds across 9 ADME assays collected over ~5 years for programs in RNA-mediated and neurodegenerative diseases, giving us a realistic program test bed.

## The data problem: why zero-shot models fall short

Before training anything, it is worth calibrating expectations. The lowest-effort approach is to use an **off-the-shelf public model** — tools like ADMETLab 3.0 or ADMET-AI are well-engineered and freely accessible. For LogD on the ExpansionRx test set, ADMETLab 3.0 gives this:

<img src="img/ADMETLab3_LogD.png" width="600">

Performance is poor even for this learnable property. There is systematic bias across the activity range, and substantial scatter. The key issue: these models are trained on broad public datasets that may not cover the specific chemical series of your program. Even our own **best-effort ChEMBL-trained model** (released on HuggingFace) improves execution but cannot close the gap:

<img src="img/OADMET_LogD.png" width="600">

And when you compare against the **ExpansionRx blind challenge** (400+ participants who trained on program data), even a simple LGBM baseline trained on program measurements handily outperforms global zero-shot models. Top of the line performance when fine tuning on program data by **Pebble** (Inductive Bio) is shown below. 

<img src="img/pebble_LogD.png" width="600">

<div style="background:light-dark(#fdecea, #3d1a15); color:light-dark(#333, #e8d0d0); border-left:5px solid #c62828; padding:14px 18px; border-radius:6px; margin:16px 0; font-size:1.1em;">
<b>⚠️ Algorithm complexity cannot rescue you from a lack of data.</b><br><br>
The data is the key lever. This is why we need rigorous pipelines for collecting and curating ADMET measurements, and why making those measurements public matters so much.
</div>

That said, the best approach when you *do* have program data is to combine public knowledge with your measurements via fine-tuning. First you need a good public-data starting point — and that is what we build below. Examples of how to fine tune are available in our [demos repo](https://github.com/OpenADMET/openadmet-demos).

## Step 1: Load curated ChEMBL LogD data

OpenADMET maintains pre-curated datasets in our [data-catalogs](https://github.com/OpenADMET/data-catalogs) repository. Each catalog entry is an `intake` data source pointing to an S3-hosted parquet file produced by our standardised curation pipeline. The curation steps (deduplication, unit harmonisation, outlier flagging, canonical SMILES generation via RDKit) are all documented and reproducible — you can re-run the exact pipeline from source.

For LogD the catalog aggregates measurements from ChEMBL 35, applying:
* Retention of measurements with `standard_type` in `{LogD, logD}` at pH 7.4
* Median aggregation across repeated measurements, with standard deviation tracked
* Removal of entries with high inter-assay variance (std > 1.5 log units)
* RDKit standardisation and InChIKey deduplication

In [ ]:
import intake
import pandas as pd

try:
    cat = intake.open_catalog(
        "https://github.com/OpenADMET/data-catalogs/raw/refs/heads/main/catalogs/activities/ChEMBL_LogD/CATALOG_ChEMBL35_LogD.yaml"
    )
    chembl_logd = cat["LogD_aggregated"].read()
except Exception:
    chembl_logd = pd.read_csv("data/chembl_logD.csv")

chembl_logd.reset_index(drop=True).head()

The aggregated dataset contains around 18,000 unique compounds. The `standard_value_mean` column is the quantity we will train to predict. Notice the `standard_value_std` column — where multiple assays agree tightly, measurements are reliable; where they diverge, the signal is noisier and the model will naturally have higher aleatoric uncertainty.

For training purposes we use a pre-filtered version (`LogD_anvil_train/logd_data.csv`) that applies one additional step: compounds with fewer than two independent assay measurements are excluded, since a single-point measurement from one lab can carry systematic assay bias.

## Step 2: Configure and train with Anvil

Much of the focus in our field is on model architectures, but **reproducibility and configurability** matter equally. A model you cannot re-train or audit is hard to trust. 

<div style="background:light-dark(#fff8e1, #2d2a1f); color:light-dark(#333, #e8e6e0); border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:16px 0;">
<b>🔨 Anvil — the OpenADMET model training harness</b><br><br>
We use <a href="https://github.com/OpenADMET/openadmet_models"><b>Anvil</b></a>, our workflow engine, to train models reproducibly. Training is fully specified by a YAML recipe — every decision about data, splitting, featurisation, architecture, and evaluation is recorded in a single file that you can version control and share. We keep canonical recipes in <a href="https://github.com/OpenADMET/optimus-prime"><b>Optimus Prime</b></a>, with hyperparameters optimised across multiple datasets.
</div>

### The recipe file

Let's look at our LogD recipe. The YAML captures every decision you would make when training:

```yaml
# --- INPUT DATA ---
data:
  resource: "{{ANVIL_DIR}}/logd_data.csv"   # path resolved relative to recipe location
  type: intake
  input_col: OPENADMET_CANONICAL_SMILES      # column containing RDKit-standardised SMILES
  target_cols:
  - logD                                     # regression target
  dropna: False

# --- METADATA ---
metadata:
  name: chemprop_logD_single_task
  authors: Maria A. Castellanos
  biotargets: [LogD]
  description: CheMeleon GNN, single-task LogD regression on curated ChEMBL data
  version: v1

# --- TRAINING PROCEDURE ---
procedure:

  # Molecular featurisation
  feat:
    type: ChemPropFeaturizer         # atom/bond features for message-passing
    params:
      batch_size: 128
      normalize_targets: true        # z-score targets before training, rescale on output
      n_jobs: 4

  # Model architecture
  model:
    type: ChemPropModel
    params:
      from_foundation: chemeleon     # initialise MPNN from CheMeleon pre-trained weights
      n_tasks: 1
      ffn_hidden_dim: 512
      ffn_hidden_num_layers: 3       # 3-layer FFN head on top of graph embedding
      dropout: 0.25
      mpnn_lr: 1e-4                  # slower LR for pre-trained message-passing layers
      ffn_lr: 1e-3                   # faster LR for the task-specific FFN head
      scheduler: plateau             # reduce LR on validation loss plateau
      reduce_lr_patience: 5

  # Train/val/test split
  split:
    type: ShuffleSplitter            # random split; ScaffoldSplitter available for harder evaluation
    params:
      random_state: 42
      train_size: 0.7
      val_size: 0.1
      test_size: 0.2

  # Training backend
  train:
    type: LightningTrainer           # PyTorch Lightning, handles GPU/CPU automatically
    params:
      accelerator: gpu
      max_epochs: 50
      early_stopping: true
      early_stopping_patience: 20
      gradient_clip_val: 0.5

# --- EVALUATION ---
report:
  eval:
  - type: RegressionMetrics          # R², MAE, RMSE, Spearman ρ with bootstrap CIs
    params: {}
  - type: RegressionPlots
    params:
      axes_labels: [True LogD, Predicted LogD]
      title: True vs Predicted LogD on test set
```

### A few things worth highlighting

**CheMeleon initialisation** (`from_foundation: chemeleon`): CheMeleon is a message-passing GNN pre-trained on millions of molecular property datapoints. Initialising from these weights rather than random initialisation gives the MPNN a broad chemical vocabulary before it sees any LogD data. The asymmetric learning rates (`mpnn_lr: 1e-4` vs `ffn_lr: 1e-3`) preserve this pre-training while training the task head quickly.

**Scaffold vs random split**: We use `ShuffleSplitter` here (random 70/10/20). For a rigorous evaluation of generalisation to new chemical series, `ScaffoldSplitter` (Bemis-Murcko scaffold-based) is the harder and more realistic choice. We recommend it for publication benchmarks.


### Run the training

The full training run takes roughly 10–15 minutes on a modern GPU. The recipe uses `accelerator: gpu` and runs for up to 50 epochs with early stopping.

The CLI interface is intentionally minimal: you give it a recipe and an output directory and it handles everything else.

In [ ]:
%set_env OADMET_NO_RICH_LOGGING=1

In [ ]:
!openadmet anvil \
    --recipe-path LogD_anvil_train/chemeleon_logd_full.yaml \
    --output-dir LogD_anvil_train/chembl_chemeleon_my_training

After training, Anvil writes the following to the output directory:

```
chembl_chemeleon_my_training/
├── model.pth             # serialised model weights (PyTorch)
├── model.json            # model config: architecture, task names, normalisation params
├── anvil_recipe.yaml     # copy of the recipe used (fully resolved paths)
├── regression_metrics.json
├── logD_regplot.png      # true vs predicted scatter on held-out test set
├── logD_ciplot.png       # bootstrap confidence interval plot
└── data/
    ├── X_train.csv / X_val.csv / X_test.csv
    └── y_train.csv / y_val.csv / y_test.csv
```

The `model.json` file is the key artefact — it records everything needed to reconstruct the model at inference time, including the featuriser config, target normalisation statistics, and task column names. This is what makes OpenADMET models self-describing and portable.

## Step 3: Run inference

With a trained model directory, the `openadmet predict` CLI handles inference. It reads `model.json` to reconstruct the correct featuriser and normalisation transform, then runs the model in batches.

We run predictions on the ExpansionRx test set using the **fully-trained model** (`chembl_chemeleon`) rather than the 1-epoch demo model.

In [ ]:
!openadmet predict \
    --input-path data/exprx_test.csv \
    --input-col OPENADMET_CANONICAL_SMILES \
    --model-dir LogD_anvil_train/chembl_chemeleon \
    --output-csv exprx_chembl_logd_predictions.csv \
    --accelerator gpu

In [2]:
import pandas as pd
import json

preds = pd.read_csv("exprx_chembl_logd_predictions.csv")

print("Prediction columns:", [c for c in preds.columns if c.startswith("OADMET")])
print(f"\nTest set size: {len(preds)} compounds")
preds[["OPENADMET_CANONICAL_SMILES", "LogD", "OADMET_PRED_chemprop_logD", "OADMET_STD_chemprop_logD"]].head()

Prediction columns: ['OADMET_PRED_chemprop_logD', 'OADMET_STD_chemprop_logD', 'OADMET_PRED_chemprop_caco2_atob_LogPapp', 'OADMET_STD_chemprop_caco2_atob_LogPapp', 'OADMET_PRED_chemprop_caco2_btoa_LogPapp', 'OADMET_STD_chemprop_caco2_btoa_LogPapp', 'OADMET_PRED_chemprop_mppb_LogUnbound', 'OADMET_STD_chemprop_mppb_LogUnbound', 'OADMET_PRED_chemprop_hppb_LogUnbound', 'OADMET_STD_chemprop_hppb_LogUnbound']

Test set size: 515 compounds


,OPENADMET_CANONICAL_SMILES,LogD,OADMET_PRED_chemprop_logD,OADMET_STD_chemprop_logD
0,CC1=CC(C2=CC(F)=C(OCCN3CCOCC3)C=C2F)=CC2=CN=CC...,4.0,3.578019,NaN
1,OC1=CC2=CC=NC=C2C=C1C1=CC(F)=C(OCCN2CCOCC2)C=C1F,2.6,2.817175,NaN
2,CCNC1=NC2=CC=NC=C2C=C1C1=CC(F)=C(OCCN2CCOCC2)C...,2.7,3.318250,NaN
3,CC1=CC2=CC=NC=C2C=C1C1=CC(F)=C(OCCN2CCOCC2)C=C1F,3.9,3.866231,NaN
4,CC1=CC(OCCN2CCCCC2)=CC(O)=C1C1=CC(C)=C2C=CN=CC...,3.2,2.948573,NaN


In [6]:
# Load the metrics computed on the training-time test split for reference
with open("LogD_anvil_train/chembl_chemeleon_my_training/regression_metrics.json") as f:
    metrics = json.load(f)

logd_metrics = metrics["logD"]
print("Performance on ChEMBL hold-out test set (training split):")
print(f"  R²       : {logd_metrics['r2']['value']:.3f}  ")
print(f"(95% CI: {logd_metrics['r2']['lower_ci']:.3f} – {logd_metrics['r2']['upper_ci']:.3f})")
print(f"  MAE      : {logd_metrics['mae']['value']:.3f}")
print(f"(95% CI: {logd_metrics['mae']['lower_ci']:.3f} – {logd_metrics['mae']['upper_ci']:.3f})")

print(f"  Spearman rho: {logd_metrics['spearmanr']['value']:.3f}")
print(f"(95% CI: {logd_metrics['spearmanr']['lower_ci']:.3f} – {logd_metrics['spearmanr']['upper_ci']:.3f})")


Performance on ChEMBL hold-out test set (training split):
  R²       : 0.626  
(95% CI: 0.604 – 0.649)
  MAE      : 0.707
(95% CI: 0.688 – 0.726)
  Spearman rho: 0.787
(95% CI: 0.773 – 0.801)


The R² of ~0.63 on the ChEMBL hold-out set is solid for a single-task model. Our **multi-task model** (training LogD alongside Caco-2 permeability and plasma protein binding, recipe in `LogD_anvil_train/chemeleon_multi.yaml`) improves this further. On the **ExpansionRx test set**, transfer from ChEMBL gives reasonable rankings but systematic bias persists. If you have program measurements, fine-tuning this model on them is the clear next step — see our [demos repo](https://demos.openadmet.org) for a full walkthrough.

---

## Step 4: Author recipes with natural language — the Anvil Claude Skill

<div style="background:light-dark(#e8f5e9, #1b2e1f); color:light-dark(#333, #d0e8d5); border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:16px 0;">
<b>🤖 openadmet-anvil-skill</b> — Anvil recipe authoring and guidance inside Claude Code<br><br>
We have built a Claude Code skill that gives Claude deep knowledge of the Anvil workflow: recipe structure, component selection, hyperparameter choices, inference, and common pitfalls. Install it once and Claude can help you write, edit, and debug Anvil recipes in plain English — and can run Anvil directly on your behalf.
<br><br>
<a href="https://github.com/OpenADMET/openadmet-anvil-skill"><b>github.com/OpenADMET/openadmet-anvil-skill</b></a>
</div>

### What it can do

The skill teaches Claude the Anvil API and also lets it execute the workflow directly in your terminal. You can:

* **Run training** — *"Train a LogD model on my_data.csv with a scaffold split"* — Claude authors the recipe and runs `openadmet anvil` for you, then summarises the results.
* **Author recipes** — *"Write me a CheMeleon multitask recipe for CYP1A2 and CYP3A4 using a scaffold split"* — Claude produces a validated YAML you can pass straight to `openadmet anvil`.
* **Edit recipes** — *"Update this recipe to use ScaffoldSplitter and increase dropout to 0.4"* — Claude modifies the YAML correctly, respecting schema constraints.
* **Guidance on inference** — *"How do I run predictions on a new CSV with the model I just trained?"* — Claude gives you the exact `openadmet predict` command with the right flags.
* **Hyperparameter advice** — *"What learning rate schedule should I use for a small fine-tuning dataset?"* — Claude draws on Optimus Prime defaults and explains the trade-offs.
* **Debug help** — paste an error from an Anvil run and Claude diagnoses configuration issues, data format problems, or environment mismatches.

### Installation

```bash
# Project-level (recommended — keeps the skill scoped to this repo)
git clone https://github.com/OpenADMET/openadmet-anvil-skill /tmp/openadmet-anvil-skill
mkdir -p .claude/skills/openadmet-anvil
cp /tmp/openadmet-anvil-skill/*.md .claude/skills/openadmet-anvil/

# Or globally (available in all your projects)
mkdir -p ~/.claude/skills/openadmet-anvil
cp /tmp/openadmet-anvil-skill/*.md ~/.claude/skills/openadmet-anvil/
```

Once installed, invoke it explicitly with the slash command or just describe what you need:

```
/openadmet-anvil write a LogD recipe with scaffold split and 50 epochs
/openadmet-anvil how do I freeze the MPNN layers for fine-tuning?
```

### Example

```
User: Write me an Anvil recipe for single-task LogD regression on my_logd.csv,
      using a scaffold split and the CheMeleon foundation model.

Claude: [produces a complete, validated YAML recipe]
        Here's the recipe. A few notes:
        - ScaffoldSplitter will give you a harder, more realistic evaluation
          than a random split — expect lower R² but better real-world signal.
        - The asymmetric LRs (mpnn_lr: 1e-4, ffn_lr: 1e-3) preserve the
          CheMeleon pre-training while quickly adapting the task head.
        - Run it with: openadmet anvil --recipe-path logd.yaml --output-dir my_logd_model
```

The skill is particularly useful for **medicinal chemists and DMPK scientists** getting started with Anvil — you describe what you want in scientific terms and get a production-ready recipe without needing to memorise the YAML schema.

---

## Step 5: The OpenADMET Prediction Portal

For the fastest possible path to predictions — with no installation, no code, and no compute to manage — we provide the **OpenADMET Prediction Portal**, a Gradio-based web application hosted on HuggingFace Spaces.

<div style="background:light-dark(#e3f2fd, #0d2137); color:light-dark(#333, #d0e8f8); border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:16px 0;">
<b>🌐 Prediction Portal</b><br><br>
<a href="https://huggingface.co/spaces/openadmet/prediction_portal"><b>huggingface.co/spaces/openadmet/prediction_portal</b></a><br><br>
Paste SMILES or upload a CSV → select endpoints → download predictions. No account required for public models.
</div>

### How to use it

1. **Navigate** to [huggingface.co/spaces/openadmet/prediction_portal](https://huggingface.co/spaces/openadmet/prediction_portal)
2. **Input your compounds** — paste SMILES strings directly, or upload a CSV with a SMILES column
3. **Select models** — choose from our publicly available endpoint models (LogD, Caco-2, HLM clearance, plasma protein binding, and more)
4. **Run predictions** — click Submit; inference typically takes a few seconds for up to a few thousand compounds
5. **Download results** — a CSV with `OADMET_PRED_*` columns, matching the format produced by `openadmet predict`

The portal runs the same `openadmet predict` pipeline under the hood, so results are identical to what you would get from the CLI.

### What models are available?

The portal exposes all models currently released on [huggingface.co/openadmet](https://huggingface.co/openadmet), including:

| Model | Endpoints | Architecture |
|---|---|---|
| [`permeability-logd-ppb-chemeleon-baseline`](https://huggingface.co/openadmet/permeability-logd-ppb-chemeleon-baseline) | LogD, Caco-2 A→B, Caco-2 B→A, mPPB, hPPB | CheMeleon multitask |
| [`microsomal-clearance-chemeleon-v1`](https://huggingface.co/openadmet/microsomal-clearance-chemeleon-v1) | HLM, RLM, MLM clearance (log CLint) | CheMeleon multitask |
| [`herg-chemeleon-baseline`](https://huggingface.co/openadmet/herg-chemeleon-baseline) | hERG pIC50 | CheMeleon single-task |
| [`cyp1a2-cyp2d6-cyp3a4-cyp2c9-chemeleon-v1`](https://huggingface.co/openadmet/cyp1a2-cyp2d6-cyp3a4-cyp2c9-chemeleon-v1) | CYP1A2, CYP2D6, CYP3A4, CYP2C9 inhibition (pIC50) | CheMeleon multitask |

All models come with uncertainty estimates — the `OADMET_STD_*` columns — which are particularly useful for flagging compounds where the model is extrapolating outside its training distribution.

<div style="background:light-dark(#fff8e1, #2d2a1f); color:light-dark(#333, #e8e6e0); border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:16px 0;">
<b>⚠️ These are public-data baselines — the same caveats apply as for any zero-shot model.</b><br><br>
As established earlier, models trained on public ChEMBL data consistently underperform on program-specific chemical series. The portal models are excellent starting points and provide much better-executed baselines than most off-the-shelf tools, but they are not a substitute for fine-tuning on your own measurements. If you have program data, we strongly recommend using these predictions as a prior and fine-tuning with Anvil — see our <a href="https://demos.openadmet.org">demos repo</a> for a full walkthrough.
</div>

---

## Step 6: Deploy the portal internally

For organisations that cannot send compound data to external servers — common in pharma and biotech due to IP considerations — we provide a **Docker image** that lets you run the portal and models entirely on-premise.

### Using models from HuggingFace via Docker

Each model on HuggingFace includes a `run_model_inference.sh` convenience script. The Docker image provides a pre-configured environment with all dependencies:

```bash
# Pull a model from HuggingFace
git clone https://huggingface.co/openadmet/permeability-logd-ppb-chemeleon-baseline
cd permeability-logd-ppb-chemeleon-baseline
git lfs install && git lfs pull  # download the actual weights

# Run predictions inside the container
docker run -it --user=root --rm \
  -v ./permeability-logd-ppb-chemeleon-baseline:/home/mambauser/model:rw \
  -v /path/to/your/data:/home/mambauser/data:ro \
  --runtime=nvidia --gpus all \
  ghcr.io/openadmet/openadmet-models:main

# Inside the container:
cd /home/mambauser/model
./run_model_inference.sh  # runs on the bundled example data

# Or run your own data:
openadmet predict \
    --input-path /home/mambauser/data/my_compounds.csv \
    --input-col SMILES \
    --model-dir /home/mambauser/model \
    --output-csv /home/mambauser/data/predictions.csv \
    --accelerator gpu
```

### Deploying the Prediction Portal internally

The portal itself is a Gradio app — it runs anywhere Docker runs. To deploy it on an internal server:

```bash
# Clone the portal
git clone https://huggingface.co/spaces/openadmet/prediction_portal
cd prediction_portal

# Build and run (models are pulled from HuggingFace on first run, then cached)
docker build -t openadmet-portal .
docker run -d \
  -p 7860:7860 \
  -v openadmet_model_cache:/root/.cache/huggingface \
  --name openadmet-portal \
  openadmet-portal

# Portal is now available at http://your-server:7860
```

After the initial model download (models are cached in the named volume), the portal operates **entirely offline** with no external network calls. This meets most internal IT security requirements for compound data.

### Bringing your own trained model

If you have trained a program-specific model with Anvil, you can mount it directly into the portal container:

```bash
docker run -d \
  -p 7860:7860 \
  -v /path/to/your/anvil/output:/models/my_logd_model:ro \
  -e CUSTOM_MODEL_DIR=/models \
  openadmet-portal
```

Your custom model then appears as an additional option in the portal's model selector alongside the public baselines — the same `model.json` self-description that makes CLI inference portable also makes it portal-compatible.

---

## Summary: the full OpenADMET stack

We have walked through the entire lifecycle of an ADMET model:

| Stage | Tool | Where |
|---|---|---|
| Data curation | `data-catalogs` intake catalog | [github.com/OpenADMET/data-catalogs](https://github.com/OpenADMET/data-catalogs) |
| Model training | `openadmet anvil` CLI | [github.com/OpenADMET/openadmet_models](https://github.com/OpenADMET/openadmet_models) |
| Inference | `openadmet predict` CLI | same |
| Canonical recipes | Optimus Prime | [github.com/OpenADMET/optimus-prime](https://github.com/OpenADMET/optimus-prime) |
| Natural language interface | Anvil Claude skill | [github.com/OpenADMET/openadmet-anvil-skill](https://github.com/OpenADMET/openadmet-anvil-skill) |
| Web interface | Prediction Portal | [huggingface.co/spaces/openadmet/prediction_portal](https://huggingface.co/spaces/openadmet/prediction_portal) |
| Internal deployment | Docker | `ghcr.io/openadmet/openadmet-models:main` |
| Public baseline models | HuggingFace | [huggingface.co/openadmet](https://huggingface.co/openadmet) |
| Documentation & tutorials | Demos site | [demos.openadmet.org](https://demos.openadmet.org) |

